# Module 2 - Index the OKF store: Qdrant (dense + BM25) and Neo4j (graph)

Start the stack first: `docker compose up --build -d`. The databases start **empty**: this notebook fills them.

| Question type | Dense vectors | BM25 | Graph |
|---|---|---|---|
| "my fan stopped spinning" (paraphrase) | good | weak | - |
| `0x4A7B_VRM_OVERVOLT` (exact token) | unreliable | excellent | - |
| "which codes light a solid red LED in the thermal subsystem?" (relations) | weak | weak | excellent |

In [ ]:
import os, sys, json, asyncio, time, logging
from pathlib import Path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)                                   # every path below is relative to the project root
os.environ.setdefault("HF_HOME", str(ROOT / "models" / "huggingface"))
logging.getLogger("neo4j").setLevel(logging.ERROR)
MODEL = os.getenv("FORMATION_MODEL", "formation-fast")   # use "formation-local" if you run LM Studio / Ollama

from backend.graph_rag.neo4j_client import Neo4jGraphClient
from backend.graph_rag.qdrant_hybrid import QdrantHybridStore
from backend.graph_rag.entity_extractor import EntityExtractor
from backend.graph_rag.chunker import chunk_okf_document, split_frontmatter
from backend.graph_rag.indexer import index_okf_store, build_chunks, okf_files
neo4j, qdrant = Neo4jGraphClient(), QdrantHybridStore()
print("Neo4j connected :", neo4j.driver is not None)
print("Qdrant ready    :", qdrant.ready, "(dense + BM25 models loaded)")

## 1. Chunking that respects structure
Tables are split **one row per chunk** (prefixed by the column names), prose is packed up to ~900 characters, and each chunk keeps its heading path and the figures it mentions.

In [ ]:
chunks = build_chunks("hardware_manuals")
print(len(chunks), "chunks in the hardware corpus;", len(build_chunks("syone_corp")), "in the Syone corpus")
row = next(c for c in chunks if c["kind"] == "table_row" and "0x4A7B_VRM_OVERVOLT" in c["text"])
print("\nA table-row chunk:\n", row["text"])
fig = next(c for c in chunks if c["images"])
print("\nA chunk that carries figures:", fig["section"], "->", [i["figure_id"] for i in fig["images"]])

## 2. Entities and relationships for the graph
Rule-based and **data-driven**: the facts (cause, remediation, LED state) are read from the document tables, not hard-coded.

In [ ]:
ex = EntityExtractor().extract_from_okf_file(ROOT / "data/okf_store/hardware_manuals/aor3_06_fault_code_reference.md")
print(len(ex["nodes"]), "nodes,", len(ex["relationships"]), "relationships from one document")
code_node = next(n for n in ex["nodes"] if n["id"] == "code:0x4A7B")
print(json.dumps(code_node, indent=1))
print([ (r["type"], r["target"]) for r in ex["relationships"] if r["source"] == "code:0x4A7B"])

## 3. Index everything (idempotent: safe to re-run)

In [ ]:
t0 = time.time()
report = index_okf_store(neo4j, qdrant)
print(json.dumps(report, indent=1)); print(f"{time.time()-t0:.0f}s")

## 4. Explore the graph with Cypher  (also open http://localhost:17474, login neo4j / syone2026)

In [ ]:
import pandas as pd
print("Node counts:"); display(pd.DataFrame(neo4j.stats()["nodes"]).T)
q = """MATCH (c:ErrorCode)-[:IN_SUBSYSTEM]->(:Subsystem {name:'Thermal'}), (c)-[:INDICATED_BY]->(l:LEDState)
       WHERE l.name CONTAINS 'Solid red' RETURN c.name AS code, c.severity AS severity, c.remediation AS remediation"""
display(pd.DataFrame(neo4j.run_cypher(q)))

In [ ]:
q = """MATCH (o:Organization)-[:PARTNERS_WITH]->(p:Partner)<-[:DESCRIBES]-(d:Document) RETURN p.name AS partner, p.tier AS tier, d.doc_id AS page ORDER BY partner"""
display(pd.DataFrame(neo4j.run_cypher(q)))

## 5. Dense vs BM25 vs hybrid, with the ranks visible
`hybrid_search` returns the rank each method gave every chunk and the fused RRF score, so you can see *why* a chunk won.

In [ ]:
def show(query, corpus, k=4):
    print(f"\nQUERY: {query}")
    for h in qdrant.hybrid_search(query, corpus=corpus, top_k=k):
        print(f"  RRF {h['hybrid_score']:.4f} | dense #{str(h['dense_rank']):>4} | bm25 #{str(h['bm25_rank']):>4} | {h['title'][:34]:34s} | {h['text'].split(chr(10),1)[1][:70]}")

show("0x4A7B_VRM_OVERVOLT", "hardware_manuals")             # the code appears in ~15 chunks: BM25 returns them all; fusion promotes the definition row
show("the cooling unit stopped rotating", "hardware_manuals")  # no shared words with the manual ("fan", "tachometer"): dense carries it, BM25 is weak
show("error 0x8F01", "hardware_manuals")                      # a generic word ("error") drags BM25 towards unrelated chunks: noise is real
show("0x4A7B", "hardware_manuals")                            # prefix only: works because we also index the split form of every code

### What the ranks tell you
* **Exact identifier** (`0x4A7B_VRM_OVERVOLT`): the code is mentioned in about 15 chunks across 8 manuals, so BM25 returns *all of them* (the definition row is not necessarily first by BM25 alone). Dense vectors know which chunk is the *definition*. Reciprocal Rank Fusion lets each method vote and the table row wins. With the full question ("What is the root cause ... for 0x4A7B_VRM_OVERVOLT?", as in the web UI) the table row is rank 1 for both methods.
* **Paraphrase** with no shared words: BM25 has nothing to match; dense finds the fan chunks.
* **Generic words hurt BM25**: `error 0x8F01` also matches every chunk containing "error". Stop-word lists and query rewriting exist for this reason.

### Near-duplicate codes: the hard case
`0x4A7B_VRM_OVERVOLT` (54.2 V, critical) vs `0x4A7C_VRM_UNDERVOLT` (40.5 V, major). Embeddings see them as almost the same text.

In [ ]:
show("0x4A7C_VRM_UNDERVOLT", "hardware_manuals", k=3)
show("Which fault means the busbar dropped below 40.5 V?", "hardware_manuals", k=3)

### Exercises
1. Query `show("0x4A7B_VRM_OVERVOLT", ...)` after replacing the BM25 sparse vector with a zero vector (hint: comment out the `bm25` prefetch in `qdrant_hybrid.py`). Which queries break?
2. Write a Cypher query that returns every code raised by the `AS-100G switch` component and the LED each one shows.
3. Add a new `Event` alias to `backend/ingestion/knowledge.py`, re-index, and find the new nodes in the Neo4j browser.